# MLflow Autologging with scikit-learn

**Notebook 1 of 3** · Demo for the "MLflow: Autologging & Zero-Config" talk

This notebook shows *true* zero-config tracking. You add one line, `mlflow.autolog()`, and MLflow records every parameter, metric, and the trained model for you.

**What you'll do**
1. Train a model the normal way (nothing recorded).
2. Turn on autologging and retrain (everything recorded).
3. Open the MLflow UI and compare two runs.

Find more at [thamu.dev](https://thamu.dev/).

## Setup

Using `uv` for package management (falls back to pip). Run this once.

In [ ]:
# Install dependencies. In a fresh environment, uncomment the line you need.
# uv:   !uv pip install "mlflow>=3.1" scikit-learn
# pip:  !pip install "mlflow>=3.1" scikit-learn

import mlflow
import sklearn
print("mlflow  :", mlflow.__version__)
print("sklearn :", sklearn.__version__)

## Step 1 — Baseline: train with no tracking

This is ordinary scikit-learn. When the kernel restarts, the result is gone. Nothing is recorded.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
clf.fit(X_tr, y_tr)

print("accuracy:", clf.score(X_te, y_te))
# A single number. Where did it come from? What settings produced it? Unrecorded.

## Step 2 — Turn on autologging

One call. `mlflow.autolog()` hooks into scikit-learn and records params, metrics, the model, and its signature automatically. We also name the experiment so runs are easy to find.

In [ ]:
# MLflow 3.x deprecated the plain-file backend, so point at a local SQLite DB.
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("sklearn-autolog-demo")
mlflow.autolog()   # <- the only MLflow line you add

clf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
clf.fit(X_tr, y_tr)   # same fit() as before — but now it's all captured

print("accuracy:", clf.score(X_te, y_te))

## Step 3 — Change a setting and rerun

Now vary a hyperparameter. Each `fit()` becomes its own tracked run, so you can compare them later.

In [ ]:
for depth in (2, 4, 8):
    model = RandomForestClassifier(n_estimators=100, max_depth=depth, random_state=42)
    model.fit(X_tr, y_tr)
    print(f"max_depth={depth}  ->  test acc {model.score(X_te, y_te):.3f}")

## Step 4 — Open the MLflow UI

Run this in a terminal (not the notebook), from the same folder:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Then open **http://localhost:5000**.

> **macOS gotcha:** port 5000 is used by AirPlay Receiver. If the UI won't load, run `mlflow ui --port 5001` (and open :5001), or turn off AirPlay Receiver in System Settings > General > AirDrop & Handoff.

In the UI you can:
- Click a run to see auto-logged **params** (n_estimators, max_depth), **metrics**, and the saved **model** artifact.
- Select two or more runs and hit **Compare** to see them side by side.

That comparison is the whole point: you can now tell which settings gave the best result.

In [ ]:
# Optional: list your runs programmatically instead of the UI
runs = mlflow.search_runs(experiment_names=["sklearn-autolog-demo"])
cols = [col for col in runs.columns
        if col.startswith("params.") or col.startswith("metrics.")][:8]
print(runs[["run_id"] + cols].to_string(index=False) if not runs.empty else "No runs yet.")

## Recap

- **One line**, `mlflow.autolog()`, turned an untracked script into a full experiment history.
- Params, metrics, model, and signature were captured with no manual logging.
- The UI lets you compare runs and pick a winner.

**Next:** Notebook 2 does the same for PyTorch, with one important catch.